# HW: Multimodal 3D Brain-MRI Registration (BraTS 2020)

## Deep Learning / Medical Image Analysis — Programming Assignment

**Big picture.** *Registration* geometrically aligns a *moving* image to a *fixed*
image so corresponding anatomy overlaps. You will build a 3D **multimodal**
registration pipeline for brain MRI with [SimpleITK](https://simpleitk.readthedocs.io/):
align one MRI modality to another, resample, and evaluate the result.

Each BraTS patient has four MRI modalities (`t1`, `t1ce`, `t2`, `flair`) plus an
expert tumor mask (`seg`). Because the modalities have different intensities,
aligning them is a genuine multimodal problem.

**Notebook input →** BraTS 2020 volumes (NIfTI `.nii`):
https://www.kaggle.com/datasets/awsaf49/brats20-dataset-training-validation
Files: `..._t1.nii`, `..._t1ce.nii`, `..._t2.nii`, `..._flair.nii`, `..._seg.nii`.
**Notebook output →** a registered volume per pair, before/after visualizations,
quantitative metrics, and PNG exports.

You are expected to **read the SimpleITK / nibabel docs and work out the details
yourself.** Each cell states only *what* a function must take in and return — the
*how* is your job.

### Deliverables & grading (100 pts)
| Part | Topic | Pts |
|------|-------|-----|
| 1 | Dataset exploration | 10 |
| 2 | Load & visualize volumes | 15 |
| 3 | Registration function | 25 |
| 4 | Run registration + resample | 10 |
| 5 | Qualitative evaluation | 15 |
| 6 | Quantitative evaluation | 10 |
| 7 | Batch registration + PNG export | 10 |
| — | Written answers (✍️) | 5 |

> Fill in every `# TODO` and ✍️ question. Keep the provided asserts and headers.
> Keep your plots/prints visible in the submission.


## Part 0 — Setup

In [ ]:
# Uncomment if a package is missing.
# !pip install SimpleITK nibabel matplotlib numpy


In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt
import SimpleITK as sitk
import nibabel as nib
np.random.seed(0)


## Part 1 — Explore the dataset  *(10 pts)*

In [ ]:
# TODO: set DATA_ROOT to the training-data folder and collect the per-patient dirs.
DATA_ROOT = "/kaggle/input/datasets/awsaf49/brats20-dataset-training-validation/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData"     # -> str path
patient_dirs = sorted([
    os.path.join(DATA_ROOT, d)
    for d in os.listdir(DATA_ROOT)
    if os.path.isdir(os.path.join(DATA_ROOT, d))
])     # -> sorted list of patient directories

assert DATA_ROOT is not None and os.path.isdir(DATA_ROOT), "DATA_ROOT not found"
assert len(patient_dirs) > 0, "No patient folders found"
print(f"{len(patient_dirs)} patients. First: {os.path.basename(patient_dirs[0])}")


In [ ]:
# TODO: get_modality_path(patient_dir, modality) -> full path to that modality's file.
#   modality in {"t1","t1ce","t2","flair","seg"}
def get_modality_path(patient_dir, modality):
    pid = os.path.basename(patient_dir.rstrip("/"))
    path = os.path.join(patient_dir, f"{pid}_{modality}.nii")
    if not os.path.exists(path):
        matches = glob.glob(os.path.join(patient_dir, f"*_{modality}.nii*"))
        if matches:
            path = matches[0]
    return path

p0 = patient_dirs[0]
for m in ["t1", "t1ce", "t2", "flair", "seg"]:
    path = get_modality_path(p0, m)
    assert path and os.path.exists(path), f"Missing {m}"
    print(f"{m:6s} -> {os.path.basename(path)}")


✍️ **Q1.1** Give one realistic scenario where two brain scans still need
registration even though BraTS modalities are already aligned.

**Your answer:** _..._

Even though the four modalities for one scan are already aligned, registration is still needed when comparing the **same patient at two different times** (e.g., before and after treatment). Each scan session has slightly different head position and orientation, so the images must be registered before comparing them to see how the tumor changed.

## Part 2 — Load & visualize a volume  *(15 pts)*

In [ ]:
# TODO: load_volume_np(path) -> (data: np.ndarray[3D float], affine: np.ndarray)
def load_volume_np(path):
    img = nib.load(path)
    data = img.get_fdata().astype(np.float32)
    return data, img.affine

vol, aff = load_volume_np(get_modality_path(p0, "t1"))
print("shape:", vol.shape); assert vol.ndim == 3


In [ ]:
# TODO: show_ortho(data, title="") -> displays the middle axial/coronal/sagittal
# slices side by side. (no return value)
def show_ortho(data, title=""):
    x, y, z = data.shape
    slices = [data[x//2, :, :], data[:, y//2, :], data[:, :, z//2]]
    labels = ["Sagittal", "Coronal", "Axial"]
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, sl, lab in zip(axes, slices, labels):
        ax.imshow(sl.T, cmap="gray", origin="lower")
        ax.set_title(f"{title} — {lab}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_ortho(vol, "Patient 0 — T1")


In [ ]:
# TODO: show the middle axial slice of all four modalities (t1,t1ce,t2,flair) in one row.
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, m in zip(axes, ["t1", "t1ce", "t2", "flair"]):
    v, _ = load_volume_np(get_modality_path(p0, m))
    ax.imshow(v[:, :, v.shape[2] // 2].T, cmap="gray", origin="lower")
    ax.set_title(m)
    ax.axis("off")
plt.tight_layout()
plt.show()

✍️ **Q2.1** Why can't `t1` be aligned to `flair` by minimizing the sum of
squared intensity differences?

**Your answer:** _`t1` and `flair` show the same tissue with **different brightness**. For example, CSF is dark in T1 but bright in FLAIR. So even when two images are perfectly aligned, matching pixel intensities would be small, not large._

## Part 3 — Registration function  *(25 pts, core)*

Implement an intensity-based registration with `sitk.ImageRegistrationMethod`.
It is multimodal, so choose your components (metric, sampling, interpolator,
optimizer, multi-resolution schedule, initial transform) accordingly. Read the
docs and decide the details yourself.

In [ ]:
# TODO: read_sitk(path) -> sitk.Image (Float32)
def read_sitk(path):
    img = sitk.ReadImage(path)
    return sitk.Cast(img, sitk.sitkFloat32)


In [ ]:
def register(fixed, moving, transform_type="rigid", verbose=True):
    """Align `moving` to `fixed`.
    Input : fixed, moving  -> sitk.Image (Float32)
            transform_type -> "rigid" or "affine"
    Output: the final sitk transform mapping moving -> fixed.
    """
    if transform_type == "rigid":
        initial_transform = sitk.CenteredTransformInitializer(
            fixed, moving, sitk.Euler3DTransform(),
            sitk.CenteredTransformInitializerFilter.MOMENTS   # CHANGED: was GEOMETRY
        )
    elif transform_type == "affine":
        initial_transform = sitk.CenteredTransformInitializer(
            fixed, moving, sitk.AffineTransform(3),
            sitk.CenteredTransformInitializerFilter.MOMENTS   # CHANGED: was GEOMETRY
        )
    else:
        raise ValueError("transform type must be rigid or affine")

    R = sitk.ImageRegistrationMethod()

    R.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
    R.SetMetricSamplingStrategy(R.RANDOM)
    R.SetMetricSamplingPercentage(0.2)
    R.SetInterpolator(sitk.sitkLinear)

    R.SetOptimizerAsRegularStepGradientDescent(
        learningRate=2.0, minStep=1e-4, numberOfIterations=300,
        relaxationFactor=0.5
    )
    R.SetOptimizerScalesFromPhysicalShift()

    R.SetShrinkFactorsPerLevel(shrinkFactors=[4, 2, 1])
    R.SetSmoothingSigmasPerLevel(smoothingSigmas=[2, 1, 0])
    R.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()

    R.SetInitialTransform(initial_transform, inPlace=False)

    if verbose:
        R.AddCommand(sitk.sitkIterationEvent, lambda: print(
            f"level {R.GetCurrentLevel()} it {R.GetOptimizerIteration():3d} "
            f"metric {R.GetMetricValue():.5f}"
        ))

    final_transform = R.Execute(fixed, moving)

    if verbose:
        print(f"Final metric value: {R.GetMetricValue():.5f}")
        print(f"Stop condition: {R.GetOptimizerStopConditionDescription()}")

    return final_transform

✍️ **Q3.1** Why does optimizing over rotation (radians) and translation (mm)
together require rescaling the parameters?

✍️ **Q3.2** What does a multi-resolution (coarse-to-fine) schedule buy you?

**Your answers:**

- Q3.1: _Rotation is measured in radians (small numbers) while translation is measured in mm (can be tens of numbers). If the optimizer treats both the same way, a small step size is too big for rotation and too small for translation, so it won't converge properly. Rescaling makes a unit change in each parameter correspond to a similar physical effect on the image, so the optimizer moves smoothly through both._
  
- Q3.2: _A coarse-to-fine schedule first aligns images at low resolution (smoothed, downsampled), which is fast and avoids getting stuck in local minima caused by fine details. It then refines the alignment at higher resolutions, using the coarse result as a good starting point. This makes registration faster and more robust to initial misalignment._


## Part 4 — Run registration & resample  *(10 pts)*

BraTS modalities are already aligned, so first apply a *known* perturbation to
create something to recover.

In [ ]:
# Provided: apply a known rigid perturbation to an sitk image.
def perturb(image, angles=(0.08, -0.05, 0.06), translation=(6.0, -4.0, 5.0)):
    tx = sitk.Euler3DTransform()
    tx.SetCenter(image.TransformContinuousIndexToPhysicalPoint(
        [(sz - 1) / 2.0 for sz in image.GetSize()]))
    tx.SetRotation(*angles); tx.SetTranslation(translation)
    return sitk.Resample(image, image, tx, sitk.sitkLinear, 0.0, image.GetPixelID()), tx


In [ ]:
# TODO:
#   fixed = read_sitk(t1 of p0); moving_aligned = read_sitk(flair of p0)
fixed = read_sitk(get_modality_path(p0, "t1"))
moving_aligned = read_sitk(get_modality_path(p0, "flair"))

#   moving, true_tx = perturb(moving_aligned)
moving, true_tx = perturb(moving_aligned)

#   estimated_tx = register(fixed, moving, "rigid")
estimated_tx = register(fixed, moving, "rigid")


#   registered = resample `moving` into `fixed`'s grid using estimated_tx
registered = sitk.Resample(moving, fixed, estimated_tx, sitk.sitkLinear, 0.0, moving.GetPixelID())

print("True transform params: ", true_tx.GetParameters())
print("Estimated transform params:", estimated_tx.GetParameters())


## Part 5 — Qualitative evaluation  *(15 pts)*

In [ ]:
# Provided: middle axial slice of an sitk image as a numpy array.
def mid_axial(sitk_img):
    arr = sitk.GetArrayFromImage(sitk_img)   # axis order (z, y, x)
    return arr[arr.shape[0] // 2]


In [ ]:
# TODO: checkerboard(a, b, tiles=8) -> np.ndarray interleaving tiles of two
# same-size 2D arrays.
def checkerboard(a, b, tiles=8):
    h, w = a.shape
    out = np.zeros_like(a)
    th, tw = h // tiles, w // tiles
    for i in range(tiles):
        for j in range(tiles):
            r0, r1 = i * th, (i + 1) * th if i < tiles - 1 else h
            c0, c1 = j * tw, (j + 1) * tw if j < tiles - 1 else w
            out[r0:r1, c0:c1] = a[r0:r1, c0:c1] if (i + j) % 2 == 0 else b[r0:r1, c0:c1]
    return out


In [ ]:
def normalize(img):
    lo, hi = np.percentile(img, [1, 99])
    return np.clip((img - lo) / (hi - lo + 1e-8), 0, 1)

fixed_n      = normalize(mid_axial(fixed))
moving_n     = normalize(mid_axial(moving))
registered_n = normalize(mid_axial(registered))

diff_before = fixed_n - moving_n
diff_after  = fixed_n - registered_n
cb_before   = checkerboard(fixed_n, moving_n, tiles=8)
cb_after    = checkerboard(fixed_n, registered_n, tiles=8)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0,0].imshow(moving_n, cmap="gray");            axes[0,0].set_title("Moving (before)")
axes[0,1].imshow(diff_before, cmap="bwr", vmin=-1, vmax=1); axes[0,1].set_title("Diff: fixed - moving")
axes[0,2].imshow(cb_before, cmap="gray");            axes[0,2].set_title("Checkerboard (before)")

axes[1,0].imshow(registered_n, cmap="gray");         axes[1,0].set_title("Registered (after)")
axes[1,1].imshow(diff_after, cmap="bwr", vmin=-1, vmax=1);  axes[1,1].set_title("Diff: fixed - registered")
axes[1,2].imshow(cb_after, cmap="gray");             axes[1,2].set_title("Checkerboard (after)")

for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()

✍️ **Q5.1** What changed between the before/after checkerboards? Any residual
misalignment?

**Your answer:** _In the before checkerboard, brain structures are visibly offset between tiles. Edges look doubled and shifted. In the after checkerboard, most structures line up smoothly across tile boundaries, and the difference image is much darker overall. There is still some small residual misalignment, especially visible as slight edge mismatch near the brain boundary likely due to the multimodal metric optimization not converging to the exact optimum._

## Part 6 — Quantitative evaluation  *(10 pts)*

In [ ]:
# TODO: metric_value(fixed, image) -> float  (a similarity metric between the two).
# Report it BEFORE (fixed vs moving) and AFTER (fixed vs registered).
def metric_value(fixed, image):
    R = sitk.ImageRegistrationMethod()
    R.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
    R.SetInterpolator(sitk.sitkLinear)
    R.SetInitialTransform(sitk.Transform())  
    return R.MetricEvaluate(fixed, image)

mi_before = metric_value(fixed, moving)
mi_after  = metric_value(fixed, registered)
print(f"MI before: {mi_before:.5f}")
print(f"MI after:  {mi_after:.5f}")


In [ ]:
# TODO: using the known true_tx, compare the true vs recovered translation and comment.
# Compare true vs recovered translation
true_params = true_tx.GetParameters()
est_params  = estimated_tx.GetParameters()

true_rot, true_trans = true_params[:3], true_params[3:]
est_rot,  est_trans  = est_params[:3],  est_params[3:]

print("True rotation (rad):     ", np.round(true_rot, 4))
print("Estimated rotation (rad):", np.round(est_rot, 4))
print("True translation (mm):     ", np.round(true_trans, 3))
print("Estimated translation (mm):", np.round(est_trans, 3))

trans_error = np.array(est_trans) - (-np.array(true_trans))
print("Translation residual vs. approx. inverse (mm):", np.round(trans_error, 3))
print("Residual magnitude (mm):", round(np.linalg.norm(trans_error), 3))

✍️ **Q6.1** Did the metric improve? Is the metric value alone enough to trust
a clinical registration — why or why not?

**Your answer:** _Yes, the metric improved: mutual information went from **-0.312 (before)** to **-0.454 (after)**(SimpleITK returns -MI; more negative = better), a clear improvement since SimpleITK minimizes this metric (more negative = better alignment). The recovered translation is also close to the true perturbation's inverse, with a residual of a few mm on one axis.

However, the metric value alone is not enough to trust a clinical registration.
A few reasons:
- MI can have local minima — a "good" MI score doesn't guarantee the anatomically correct alignment, especially with pathology (tumors) that can create high-MI matches at wrong poses.
- MI is a global, unitless statistic — it doesn't tell you where residual misalignment is (e.g. a 3mm shift could be negligible for gross anatomy but critical near a tumor boundary used for surgical planning).
- Clinically, you need spatially interpretable checks too: visual inspection (checkerboards, overlays), landmark or segmentation-based distance metrics not just a single scalar score._

## Part 7 — Batch registration + PNG export  *(10 pts)*

In [ ]:
OUTPUT_DIR = "/kaggle/working/registered"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# TODO: nifti_to_png(sitk_img, out_path) -> saves the middle slice normalized to
# uint8 [0,255] as a PNG. (no return)
def nifti_to_png(sitk_img, out_path):
    arr = mid_axial(sitk_img)
    lo, hi = np.percentile(arr, [1, 99])
    arr_n = np.clip((arr - lo) / (hi - lo + 1e-8), 0, 1)
    arr_u8 = (arr_n * 255).astype(np.uint8)
    plt.imsave(out_path, arr_u8, cmap="gray")


In [ ]:
# TODO: for the first N=3 patients, register flair->t1, write the registered
# NIfTI and a PNG of the middle slice, printing one status line each.
N = 3

for i in range(N):
    pdir = patient_dirs[i]
    pid = os.path.basename(pdir)

    fixed_i = read_sitk(get_modality_path(pdir, "t1"))
    moving_i = read_sitk(get_modality_path(pdir, "flair"))

    tx_i = register(fixed_i, moving_i, "rigid", verbose=False)
    registered_i = sitk.Resample(
        moving_i, fixed_i, tx_i, sitk.sitkLinear, 0.0, moving_i.GetPixelID()
    )

    nii_path = os.path.join(OUTPUT_DIR, f"{pid}_flair2t1.nii.gz")
    png_path = os.path.join(OUTPUT_DIR, f"{pid}_flair2t1.png")

    sitk.WriteImage(registered_i, nii_path)
    nifti_to_png(registered_i, png_path)

    mi_i = metric_value(fixed_i, registered_i)
    print(f"[{i+1}/{N}] {pid} — registered, MI(after): {mi_i:.5f} -> saved {os.path.basename(nii_path)}, {os.path.basename(png_path)}")

✍️ **Q7.1 (Reflection)** Name two concrete changes you would try if a
registration diverged, and why.

Two concrete changes if a registration diverged:

1. **Switch the centering initializer from GEOMETRY to MOMENTS** (or try a manual initial alignment). If the initial transform starts too far from the true alignment, gradient-based optimizers can converge to a wrong local minimum instead of the global one. MOMENTS centers on the intensity-weighted center of mass, which is usually a much better starting guess for brain MRI than the raw bounding-box center.

2. **Reduce the learning rate / increase the multi-resolution coarse levels** (e.g. add a coarser shrink factor like 8, or lower learningRate). A step size that's too large for the image's scale can cause the optimizer to overshoot and oscillate instead of converging, especially at the finest resolution level. Starting coarser and taking smaller, more controlled steps makes the descent more stable and less likely to diverge.

### Bonus (+10)
- Affine vs rigid on the same pair.
- Warp the `seg` mask and report Dice.
- Add a BSpline deformable stage.

In [ ]:
# Rigid vs Affine on the same pair (fixed = t1, moving = perturbed flair)

rigid_tx = register(fixed, moving, "rigid", verbose=False)
affine_tx = register(fixed, moving, "affine", verbose=False)

registered_rigid = sitk.Resample(moving, fixed, rigid_tx, sitk.sitkLinear, 0.0, moving.GetPixelID())
registered_affine = sitk.Resample(moving, fixed, affine_tx, sitk.sitkLinear, 0.0, moving.GetPixelID())

mi_rigid = metric_value(fixed, registered_rigid)
mi_affine = metric_value(fixed, registered_affine)

print(f"MI rigid:  {mi_rigid:.5f}")
print(f"MI affine: {mi_affine:.5f}")

fixed_n          = normalize(mid_axial(fixed))
reg_rigid_n      = normalize(mid_axial(registered_rigid))
reg_affine_n     = normalize(mid_axial(registered_affine))

cb_rigid  = checkerboard(fixed_n, reg_rigid_n, tiles=8)
cb_affine = checkerboard(fixed_n, reg_affine_n, tiles=8)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(cb_rigid, cmap="gray");  axes[0].set_title(f"Rigid (MI={mi_rigid:.4f})")
axes[1].imshow(cb_affine, cmap="gray"); axes[1].set_title(f"Affine (MI={mi_affine:.4f})")
for ax in axes: ax.axis("off")
plt.tight_layout()
plt.show()

**Your answer / bonus notes:** _..._